# Project 4: Model arena

This notebook is the starting point of the project; the subject (`projects.pdf`, section 4)
gives the context and the grading. You hand in this notebook, completed: it must run from
top to bottom in less than 15 minutes on a laptop.

The project has three guided tasks, then a part where you choose how to go further:

1. **Uniform duels**: find the $3$ best models from duels between random pairs, with two
   estimators.
2. **Choosing the duels**: spend the second half of the budget on the models that matter.
3. **When Bradley-Terry is wrong**: measure what happens when some preferences are cyclic.
4. **Going further**: a direction of your choice (see the end of the notebook).

Each task ends with a figure and a short written answer (a few sentences) in the cell
"Answer". Everything outside the cells marked `...  # TODO` is given and working.

This notebook needs the version of `seqlearn` that contains `projects/`: if the imports below
fail, download the setup archive again from the course page and replace the folder
`seqlearn/`.

In [ ]:
# Checks that your folder seqlearn/ is recent enough, and updates it otherwise. (Provided, just run it.)
import importlib.util, io, pathlib, re, shutil, sys, urllib.request, zipfile

REQUIRED = "2026.10.30"
URL = "https://igorcolin.github.io/teaching/sequential-decision/sequential-decision-setup.zip"


def seqlearn_version():
    spec = importlib.util.find_spec("seqlearn")
    if spec is None or spec.origin is None:
        return None, pathlib.Path("seqlearn")
    init = pathlib.Path(spec.origin)
    match = re.search(r'^__version__ = "([^"]+)"', init.read_text(), re.MULTILINE)
    return (match.group(1) if match else "none"), init.parent


version, folder = seqlearn_version()
if version is not None and version != "none" and version >= REQUIRED:
    print(f"seqlearn {version}: up to date")
else:
    print(f"seqlearn {version or 'not found'}: version {REQUIRED} or later needed, downloading it")
    try:
        data = urllib.request.urlopen(URL, timeout=30).read()
    except OSError as error:
        raise RuntimeError(f"download failed ({error}): download {URL} by hand, unzip it, "
                           "and replace your folder seqlearn/ with the new one") from None
    old = folder.with_name("seqlearn.old")
    if folder.exists():
        shutil.rmtree(old, ignore_errors=True)
        folder.rename(old)
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        for name in archive.namelist():
            parts = pathlib.PurePosixPath(name).parts
            if len(parts) > 2 and parts[1] == "seqlearn" and not name.endswith("/"):
                target = folder.joinpath(*parts[2:])
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(name))
    importlib.invalidate_caches()
    version = seqlearn_version()[0]
    if version == "none" or version < REQUIRED:
        raise RuntimeError(f"the course page still has seqlearn {version}: tell your teacher")
    print(f"seqlearn updated to {version} (previous folder kept as {old})")
    if "seqlearn" in sys.modules:
        print("Restart the kernel (Kernel > Restart), then run the notebook from the top.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from seqlearn.plotting import COLORS, set_style
from seqlearn.preferences import kendall_tau, record, sigmoid
from seqlearn.projects.arena import N_MODELS, ProjectArena

set_style()

## 0. The arena

An arena ranks $20$ language models from duels judged by annotators: `arena.duel(i, j)`
returns $1$ if model $i$ wins. The probability that $i$ beats $j$ is a Bradley-Terry
probability (session 6), perturbed by a term that makes some preferences **cyclic** (A beats
B, B beats C, C beats A), and $10\%$ of the annotators answer at random.

The quality of a model is its **Borda score**: its probability of beating an opponent drawn
uniformly among the $19$ others (`arena.borda`, for the evaluation only). The goal is to
announce $3$ models whose Borda scores are all at least the third best Borda score minus
$\varepsilon = 0.02$ (`is_good` checks it), with as few duels as possible.

In [ ]:
arena = ProjectArena(rng=0)
print("Borda scores:", np.round(np.sort(arena.borda)[::-1], 3))
print("one duel between models 0 and 1:", arena.duel(0, 1))

## Given: estimators and evaluation

The results of the duels are stored in a matrix `wins`, where `wins[i, j]` counts the wins of
$i$ against $j$ (`record(wins, i, j, i_wins)`, as in Lab 6). Two estimators of the quality
of the models:

- `win_rates(wins)`: the proportion of duels won by each model, an unbiased estimate of its
  Borda score when its opponents are drawn uniformly;
- `fit_bradley_terry(wins)`: the Bradley-Terry scores by maximum likelihood (Lab 6).

A strategy is a function `strategy(arena, budget, rng)` that plays `budget` duels and
returns the matrix `wins`. `success_rate` runs a strategy on $100$ arenas and returns, for
each estimator, the proportion of arenas where its top $3$ is good.

In [ ]:
EPS = 0.02


def is_good(arena, top3):
    third = np.sort(arena.borda)[-3]
    return all(arena.borda[i] >= third - EPS for i in top3)


def win_rates(wins):
    n = (wins + wins.T).sum(axis=1)
    return wins.sum(axis=1) / np.maximum(n, 1)


def fit_bradley_terry(wins, lam=0.1, n_steps=300):
    n = wins + wins.T
    scores = np.zeros(len(wins))
    step = 1 / max(1.0, n.sum(axis=1).max() / 4 + lam)
    for _ in range(n_steps):
        gradient = wins.sum(axis=1) - (n * sigmoid(scores[:, None] - scores[None, :])).sum(axis=1)
        scores += step * (gradient - lam * scores)
    return scores


def top3(values):
    return list(np.argsort(-values)[:3])


ESTIMATORS = {"win rates": win_rates, "Bradley-Terry": fit_bradley_terry}


N_ARENAS = 100


def success_rate(strategy, budget, n_arenas=N_ARENAS, intransitivity=0.6):
    hits = {name: [] for name in ESTIMATORS}
    for seed in range(n_arenas):
        arena = ProjectArena(rng=seed, intransitivity=intransitivity)
        wins = strategy(arena, budget, np.random.default_rng(1000 + seed))
        assert arena.n_duels <= budget
        for name, estimator in ESTIMATORS.items():
            hits[name].append(is_good(arena, top3(estimator(wins))))
    return {name: np.mean(h) for name, h in hits.items()}


def plot_success(results, budgets, title):
    """results: dictionary (strategy, estimator) -> list of success rates, one per budget."""
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    for k, (name, rates) in enumerate(results.items()):
        rates = np.array(rates)
        se = np.sqrt(rates * (1 - rates) / N_ARENAS)
        ax.errorbar(budgets, rates, yerr=2 * se, fmt="o-", color=COLORS[k % len(COLORS)], capsize=3, label=name)
    ax.set(xscale="log", xlabel="number of duels", ylabel="proportion of good top 3", ylim=(0, 1.05), title=title)
    ax.set_xticks(budgets, [str(b) for b in budgets])
    ax.minorticks_off()
    ax.legend(loc="lower right")
    plt.show()

## Task 1: uniform duels

Complete `uniform_duels`: each duel opposes two distinct models drawn uniformly
(`rng.choice(N_MODELS, size=2, replace=False)`). Then compute the success rates of the two
estimators for budgets of $500$, $1000$, $3000$ and $10\,000$ duels, and plot them.

**Expected**: the figure, and an answer to: how many duels are needed to find a good top $3$
in $90\%$ of the arenas? Is one estimator clearly better? (Look at the error bars.)

**To read**: slides of session 6 (Bradley-Terry, maximum likelihood); Chiang et al.,
*Chatbot Arena: an open platform for evaluating LLMs by human preference*, ICML 2024
(sections 3 and 4).

In [ ]:
def uniform_duels(arena, budget, rng):
    wins = np.zeros((N_MODELS, N_MODELS))
    ...  # TODO
    return wins

In [ ]:
budgets = [500, 1000, 3000, 10_000]
results1 = {}
...  # TODO
plot_success(results1, budgets, "Task 1")

**Answer.** *Write your answer here.*

## Task 2: choosing the duels

Most uniform duels involve models that are clearly not in the top $3$. Complete
`focused_duels`: spend half of the budget on uniform duels; then, in $4$ rounds of
`budget // 8` duels, take the $8$ models with the best current win rates and make one of them,
drawn uniformly, play against an opponent drawn uniformly among the $19$ others. (The
opponent stays uniform, so that the win rates still estimate the Borda scores.)

Compare with Task 1 on the same budgets.

**Expected**: the figure, and an answer to: does the focus help, for which budgets, and is
the difference significant given the error bars? Why does the strategy keep $8$ candidates
and not $3$? Relate it to sequential
halving and successive elimination (session 6).

**To read**: slides of session 6 (sequential halving, successive elimination);
Kalyanakrishnan, Tewari, Auer and Stone, *PAC subset selection in stochastic multi-armed
bandits*, ICML 2012 (sections 1 and 2).

In [ ]:
def focused_duels(arena, budget, rng, n_candidates=8, n_rounds=4):
    wins = uniform_duels(arena, budget // 2, rng)
    ...  # TODO
    return wins

In [ ]:
results2 = dict(results1)
...  # TODO
plot_success(results2, budgets, "Task 2")

**Answer.** *Write your answer here.*

## Task 3: when Bradley-Terry is wrong

The parameter `intransitivity` of `ProjectArena` sets the weight of the cyclic preferences
($0.6$ by default). For `intransitivity` in $\{0, 0.6, 1.5, 3\}$, on $20$ arenas, play
$10\,000$ uniform duels and compute:

- the error of the model: the mean of $|\sigma(\hat s_i - \hat s_j) - P_{ij}|$ over the pairs
  $i \neq j$, where $\hat s$ are the fitted Bradley-Terry scores and `arena.P` the true
  probabilities;
- the Kendall correlation with the Borda scores (`kendall_tau`) of the Bradley-Terry scores,
  and of the win rates.

**Expected**: a figure with these three quantities against `intransitivity`, and an answer
to: what does Bradley-Terry fail to represent, and which ranking should be trusted when the
preferences are cyclic? Is the "best model" still well defined?

**To read**: Bengs, Busa-Fekete, El Mesaoudi-Paul and Hüllermeier, *Preference-based online
learning with dueling bandits: a survey*, JMLR 2021 (section 3, on the notions of winner).

In [ ]:
levels = [0, 0.6, 1.5, 3]
model_error, tau_bt, tau_wr = [], [], []
...  # TODO
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(levels, model_error, "o-", color=COLORS[0])
axes[0].set(xlabel="intransitivity", ylabel="mean |predicted - true probability|", title="Task 3: error of Bradley-Terry")
axes[1].plot(levels, tau_bt, "o-", color=COLORS[0], label="Bradley-Terry scores")
axes[1].plot(levels, tau_wr, "o-", color=COLORS[1], label="win rates")
axes[1].set(xlabel="intransitivity", ylabel="Kendall correlation with Borda", title="Task 3: rankings")
axes[1].legend()
plt.show()

**Answer.** *Write your answer here.*

## Going further

Choose one direction, among these suggestions or your own idea: state the question you ask,
then answer it as in the tasks (code, a figure, a short answer). One well-studied direction
is better than several superficial ones.

- A stopping rule: confidence intervals on the win rates (Hoeffding, session 1), and stop as
  soon as $3$ models are surely good; count the duels used.
- Thompson sampling on the Bradley-Terry scores to choose the pairs.
- Other notions of winner when the preferences are cyclic: Copeland, the von Neumann winner
  (Bengs et al., section 3).